In [ ]:
import sys, os, importlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
import pathlib as _pl

# Locate repo root heuristically
_cwd = _pl.Path.cwd()
_repo_root = None
for parent in [_cwd] + list(_cwd.parents):
    if (parent / 'README.md').exists() and (parent / 'Feature_extraction').exists():
        _repo_root = parent
        break
_repo_root = _repo_root or _cwd
for p in [str(_repo_root), str(_repo_root / 'Model_Calibration')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Import demo module
try:
    demo = importlib.import_module('Model_Calibration.demo_adjust_fit_events')
except Exception:
    demo = importlib.import_module('demo_adjust_fit_events')

DEFAULT_IN_DIR = r"C:\\Users\\Antoine.Valera\\Desktop\\PPR_DATA_FINAL\\Theo_1_5Ca\\"

def _scan_argv_for_dir(argv):
    for a in argv[1:]:
        if not a or a.startswith('-'):
            continue
        ap = os.path.abspath(a)
        if os.path.isdir(ap):
            return ap
    return None

def _resolve_input_dir():
    scan = _scan_argv_for_dir(sys.argv)
    if scan:
        return scan
    env_dir = os.environ.get('GLUSNFR_IN_DIR')
    if env_dir and os.path.isdir(env_dir):
        return env_dir
    return DEFAULT_IN_DIR

USER_DIR = _resolve_input_dir()
print(f"[notebook] Using data directory: {USER_DIR}")

# Single call only
RESULTS = demo.process_folder(USER_DIR, max_workers=getattr(demo, 'MAX_WORKERS', 24))
if not RESULTS:
    raise RuntimeError('No results returned; verify directory and Excel files.')

print(f"Loaded {RESULTS['n_files']} files; time grid length = {len(RESULTS['time_grid'])}")

# shift Results timescale once for all by 3ms
RESULTS['time_grid'] += 0.003

[notebook] Using data directory: C:\\Users\\Antoine.Valera\\Desktop\\PPR_DATA_FINAL\\Theo_1_5Ca\\
Found 21 valid Excel files in C:\\Users\\Antoine.Valera\\Desktop\\PPR_DATA_FINAL\\Theo_1_5Ca\\
Processing files with 24 workers...


In [ ]:
# =============================================================================
# CELL 2: Plot the data 
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt

# Extract data from RESULTS (new structure)
t_grid = RESULTS['time_grid'] * 1000  # Convert to ms
traces = RESULTS['traces']
y_avg = RESULTS['average']
n_files = RESULTS['n_files']

# Create plot
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

# Plot individual traces (show first 20 for visibility)
max_show = min(20, len(traces))
for i, trace in enumerate(traces[:max_show]):
    alpha = 0.4 if len(traces) <= 10 else 0.2  # Adjust transparency based on number of traces
    ax.plot(t_grid, trace, 'gray', alpha=alpha, linewidth=0.8)

# Plot average trace
ax.plot(t_grid, y_avg, 'red', linewidth=3, label=f'Average (n={n_files})')

# Add stimulus marker
ax.axvline(0, color='black', linestyle='--', alpha=0.7, linewidth=1.5, label='Stimulus')

# Formatting
ax.set_xlabel('Time (ms)')
ax.set_ylabel('ΔF')
if demo.EVENT_INDEX is not None:
    title = f'Calcium Response - Event {demo.EVENT_INDEX + 1} Only'
else:
    title = f'Calcium Response - All Events Combined'
ax.set_title(title)
ax.legend()
ax.grid(True, alpha=0.3)

# Add some stats as text
stats_text = f'Files: {n_files}\nTime range: {t_grid[0]:.1f} to {t_grid[-1]:.1f} ms'
ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

# Print summary stats
print(f"Processed {n_files} files")
print(f"Time range: {t_grid[0]:.1f} to {t_grid[-1]:.1f} ms")
print(f"Peak average response: {np.nanmax(y_avg):.4f}")
if demo.EVENT_INDEX is not None:
    print(f"Analyzing event {demo.EVENT_INDEX + 1} only")
else:
    print("Analyzing all events combined")

In [ ]:
# =============================================================================
# CELL 3A: Setup and Data Preparation for Synaptic Current Analysis
# =============================================================================

import numpy as np
from scipy.optimize import curve_fit, nnls
from scipy import stats
import matplotlib.pyplot as plt

class SynapticCurrentAnalyzer:
    def __init__(self):
        self.models = {}
        
    def clean_trace(self, trace):
        """Remove NaN values and interpolate if needed"""
        trace = np.array(trace, dtype=float)
        
        if np.all(np.isnan(trace)):
            return None
            
        valid_mask = ~np.isnan(trace)
        if np.sum(valid_mask) < 5:
            return None
            
        if not np.all(valid_mask):
            x = np.arange(len(trace))
            trace_clean = np.interp(x, x[valid_mask], trace[valid_mask])
        else:
            trace_clean = trace
            
        return trace_clean
    
    def baseline_correct(self, time_ms, trace):
        """Baseline correction using pre-stimulus data"""
        baseline_mask = time_ms < 0
        
        if not np.any(baseline_mask):
            n_baseline = max(1, len(time_ms) // 4)
            baseline_mask = np.zeros(len(time_ms), dtype=bool)
            baseline_mask[:n_baseline] = True
            
        baseline_data = trace[baseline_mask]
        baseline_data = baseline_data[np.isfinite(baseline_data)]
        
        if len(baseline_data) == 0:
            return trace, 0
            
        F0 = np.median(baseline_data)
        return trace - F0, F0
    
    def estimate_noise_level(self, time_ms, trace):
        """Estimate noise level from baseline period"""
        baseline_mask = time_ms < 0
        if not np.any(baseline_mask):
            baseline_mask = np.zeros(len(time_ms), dtype=bool)
            baseline_mask[:len(time_ms)//4] = True
            
        baseline_data = trace[baseline_mask]
        baseline_data = baseline_data[np.isfinite(baseline_data)]
        
        if len(baseline_data) < 3:
            return 0.001  # Default small value
            
        return np.std(baseline_data)

# Prepare data
analyzer = SynapticCurrentAnalyzer()

# Extract data from RESULTS (updated variable names)
time_grid = RESULTS['time_grid'] * 1000  # Convert to ms
traces = RESULTS['traces']

# Apply analysis window (-5ms to 50ms for better model fitting)
analysis_mask = (time_grid >= -5) & (time_grid <= 50)
time_analysis = time_grid[analysis_mask]

# Process traces
clean_traces = []
noise_levels = []

print("=== DATA PREPARATION ===")
print(f"Analysis window: {time_analysis[0]:.1f} to {time_analysis[-1]:.1f} ms")

for i, trace in enumerate(traces):
    trace_clean = analyzer.clean_trace(trace)
    if trace_clean is None:
        continue
        
    trace_cut = trace_clean[analysis_mask]
    trace_corrected, f0 = analyzer.baseline_correct(time_analysis, trace_cut)
    
    if np.all(np.isnan(trace_corrected)):
        continue
        
    # Estimate noise level for this trace
    noise_level = analyzer.estimate_noise_level(time_analysis, trace_corrected)
    
    clean_traces.append(trace_corrected)
    noise_levels.append(noise_level)

print(f"Successfully processed: {len(clean_traces)}/{len(traces)} traces")

if len(clean_traces) == 0:
    print("ERROR: No valid traces found!")
else:
    # Calculate average trace and average noise level
    traces_array = np.array(clean_traces)
    y_avg = np.mean(traces_array, axis=0)
    avg_noise = np.mean(noise_levels)
    
    print(f"Average signal range: {y_avg.min():.4f} to {y_avg.max():.4f}")
    print(f"Average noise level: {avg_noise:.4f}")
    
    # Quick visualization
    plt.figure(figsize=(10, 4))
    for trace in traces_array[:10]:  # Show first 10
        plt.plot(time_analysis, trace, 'gray', alpha=0.3, linewidth=0.8)
    plt.plot(time_analysis, y_avg, 'red', linewidth=2, label=f'Average (n={len(clean_traces)})')
    plt.axvline(0, color='k', linestyle='--', alpha=0.7)
    plt.axhline(0, color='k', linestyle='-', alpha=0.3)
    plt.xlabel('Time (ms)')
    plt.ylabel('Signal')
    plt.title('Processed Data for Model Fitting')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# =============================================================================
# CELL 3B: Complete Model Library - Simplest to Most Complex
# =============================================================================

# =============================================================================
# 3-PARAMETER MODELS (Simplest)
# =============================================================================

def model_single_exp_constrained(t, amp, tau_decay, t_peak):
    """Single exponential decay (instantaneous rise)"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        result[mask] = amp * np.exp(-t_shifted / max(tau_decay, 1e-6))
    
    return result

def model_alpha_constrained(t, amp, tau, t_peak):
    """Alpha function: t*exp(-t/tau) - single time constant"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        tau_safe = max(tau, 1e-6)
        
        e_inv = 1.0 / np.e
        result[mask] = amp * (t_shifted / tau_safe) * np.exp(-t_shifted / tau_safe) / e_inv
    
    return result

# =============================================================================
# 4-PARAMETER MODELS
# =============================================================================

def model_double_exp_constrained(t, amp, tau_rise, tau_decay, t_peak):
    """Classic double exponential: (exp(-t/tau_decay) - exp(-t/tau_rise))"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        
        tau_rise = max(tau_rise, 1e-6)
        tau_decay = max(tau_decay, 1e-6)
        
        if tau_decay > tau_rise:
            t_opt = tau_rise * tau_decay / (tau_decay - tau_rise) * np.log(tau_decay / tau_rise)
            norm_factor = np.exp(-t_opt / tau_decay) - np.exp(-t_opt / tau_rise)
            
            rise_term = np.exp(-t_shifted / tau_rise)
            decay_term = np.exp(-t_shifted / tau_decay)
            
            if norm_factor > 1e-10:
                result[mask] = amp * (decay_term - rise_term) / norm_factor
            else:
                result[mask] = amp * (decay_term - rise_term)
    return result

def model_gamma_constrained(t, amp, n, tau, t_peak):
    """Gamma function: t^n * exp(-t/tau) - generalized alpha"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        tau_safe = max(tau, 1e-6)
        n_safe = max(n, 0.1)
        
        normalized_t = t_shifted / tau_safe
        
        try:
            norm_factor = (n_safe / np.e) ** n_safe
            result[mask] = amp * (normalized_t ** n_safe) * np.exp(-normalized_t) / norm_factor
        except:
            result[mask] = amp * (normalized_t ** n_safe) * np.exp(-normalized_t)
    
    return result

def model_bilinear_constrained(t, amp, t_rise, t_decay, t_peak):
    """Bilinear: linear rise + exponential decay"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    # Rise phase: linear from 0 to amp
    rise_mask = (t >= t_peak) & (t <= t_peak + t_rise)
    if np.any(rise_mask):
        t_rel = t[rise_mask] - t_peak
        result[rise_mask] = amp * (t_rel / max(t_rise, 0.1))
    
    # Decay phase: exponential decay from amp to 0
    decay_mask = t > (t_peak + t_rise)
    if np.any(decay_mask):
        t_rel = (t[decay_mask] - t_peak - t_rise) / 1000
        tau_decay_s = max(t_decay / 1000, 1e-6)
        result[decay_mask] = amp * np.exp(-t_rel / tau_decay_s)
    
    return result

# =============================================================================
# 5-PARAMETER MODELS
# =============================================================================

def model_cooperative_binding(t, amp, tau_rise, tau_decay, n_coop, t_peak):
    """Cooperative binding: Hill-like rise + exponential decay"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        
        # Cooperative rise phase
        tau_rise_safe = max(tau_rise, 1e-6)
        n_safe = max(n_coop, 0.5)
        
        # Hill-like binding kinetics
        normalized_t = t_shifted / tau_rise_safe
        rise_factor = (normalized_t ** n_safe) / (1 + normalized_t ** n_safe)
        
        # Exponential decay from bound state
        decay_factor = np.exp(-t_shifted / max(tau_decay, 1e-6))
        
        result[mask] = amp * rise_factor * decay_factor
    
    return result

def model_binding_kinetics(t, amp, kon, koff, tau_clear, t_peak):
    """Explicit binding/unbinding with clearance"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        
        # Binding approach to equilibrium
        kon_safe = max(kon, 1)
        binding = 1 - np.exp(-t_shifted * kon_safe)
        
        # Unbinding + clearance
        koff_safe = max(koff, 1)
        decay_total = np.exp(-t_shifted * (koff_safe + 1/max(tau_clear, 1e-6)))
        
        result[mask] = amp * binding * decay_total
    
    return result

# =============================================================================
# 6-PARAMETER MODELS
# =============================================================================

def model_two_component_shared_rise(t, amp_fast, tau_rise, tau_fast, amp_slow, tau_slow, t_peak):
    """Two-component: shared exponential rise, fast+slow decay"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        
        # Shared rise phase
        rise_term = 1 - np.exp(-t_shifted / max(tau_rise, 1e-6))
        
        # Two decay components
        fast_decay = amp_fast * np.exp(-t_shifted / max(tau_fast, 1e-6))
        slow_decay = amp_slow * np.exp(-t_shifted / max(tau_slow, 1e-6))
        
        result[mask] = rise_term * (fast_decay + slow_decay)
    
    return result

def model_desensitization(t, amp, tau_rise, tau_decay, tau_recovery, desens_factor, t_peak):
    """Desensitization: rise-decay with slow recovery component"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        
        # Basic rise-decay
        rise = 1 - np.exp(-t_shifted / max(tau_rise, 1e-6))
        decay = np.exp(-t_shifted / max(tau_decay, 1e-6))
        
        # Desensitization/recovery component
        recovery = 1 - desens_factor * (1 - np.exp(-t_shifted / max(tau_recovery, 1e-6)))
        
        result[mask] = amp * rise * decay * recovery
    
    return result

# =============================================================================
# 7-PARAMETER MODELS
# =============================================================================

def model_cooperative_plus_linear(t, amp_coop, tau_rise_coop, tau_decay_coop, n_coop,
                                 amp_linear, tau_decay_linear, t_peak):
    """Cooperative + non-cooperative binding modes"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        
        # Cooperative component
        tau_rise_safe = max(tau_rise_coop, 1e-6)
        n_safe = max(n_coop, 0.5)
        norm_t = t_shifted / tau_rise_safe
        rise_coop = (norm_t ** n_safe) / (1 + norm_t ** n_safe)
        decay_coop = np.exp(-t_shifted / max(tau_decay_coop, 1e-6))
        cooperative_part = amp_coop * rise_coop * decay_coop
        
        # Linear (non-cooperative) component
        rise_linear = 1 - np.exp(-t_shifted / tau_rise_safe)
        decay_linear = np.exp(-t_shifted / max(tau_decay_linear, 1e-6))
        linear_part = amp_linear * rise_linear * decay_linear
        
        result[mask] = cooperative_part + linear_part
    
    return result

def model_diffusion_clearance(t, amp, tau_diff, tau_clear1, tau_clear2, frac_clear1, t_peak):
    """Diffusion-limited rise with dual clearance mechanisms"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        
        # Diffusion-limited rise (alpha function)
        tau_diff_safe = max(tau_diff, 1e-6)
        rise_part = (t_shifted / tau_diff_safe) * np.exp(-t_shifted / tau_diff_safe)
        
        # Dual clearance mechanisms
        clear1 = frac_clear1 * np.exp(-t_shifted / max(tau_clear1, 1e-6))
        clear2 = (1 - frac_clear1) * np.exp(-t_shifted / max(tau_clear2, 1e-6))
        
        # Normalize rise part
        e_inv = 1.0 / np.e
        rise_normalized = rise_part / e_inv
        
        result[mask] = amp * rise_normalized * (clear1 + clear2)
    
    return result

# =============================================================================
# 8-PARAMETER MODELS
# =============================================================================

def model_double_cooperative(t, amp, tau_rise1, tau_decay1, n1, tau_rise2, tau_decay2, n2, t_peak):
    """Two cooperative components with different kinetics"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        
        # First cooperative component
        tau_rise1_safe = max(tau_rise1, 1e-6)
        n1_safe = max(n1, 0.5)
        norm_t1 = t_shifted / tau_rise1_safe
        rise1 = (norm_t1 ** n1_safe) / (1 + norm_t1 ** n1_safe)
        decay1 = np.exp(-t_shifted / max(tau_decay1, 1e-6))
        component1 = 0.5 * rise1 * decay1
        
        # Second cooperative component
        tau_rise2_safe = max(tau_rise2, 1e-6)
        n2_safe = max(n2, 0.5)
        norm_t2 = t_shifted / tau_rise2_safe
        rise2 = (norm_t2 ** n2_safe) / (1 + norm_t2 ** n2_safe)
        decay2 = np.exp(-t_shifted / max(tau_decay2, 1e-6))
        component2 = 0.5 * rise2 * decay2
        
        result[mask] = amp * (component1 + component2)
    
    return result

# =============================================================================
# 9-PARAMETER MODELS (Most Complex)
# =============================================================================

def model_heterogeneous_cooperative(t, amp, tau_rise1, tau_decay1, n1, frac1,
                                   tau_rise2, tau_decay2, n2, t_peak):
    """Heterogeneous sensor populations with different cooperativities"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        
        # Population 1
        tau_rise1_safe = max(tau_rise1, 1e-6)
        n1_safe = max(n1, 0.5)
        norm_t1 = t_shifted / tau_rise1_safe
        rise1 = (norm_t1 ** n1_safe) / (1 + norm_t1 ** n1_safe)
        decay1 = np.exp(-t_shifted / max(tau_decay1, 1e-6))
        component1 = frac1 * rise1 * decay1
        
        # Population 2
        tau_rise2_safe = max(tau_rise2, 1e-6)
        n2_safe = max(n2, 0.5)
        norm_t2 = t_shifted / tau_rise2_safe
        rise2 = (norm_t2 ** n2_safe) / (1 + norm_t2 ** n2_safe)
        decay2 = np.exp(-t_shifted / max(tau_decay2, 1e-6))
        component2 = (1 - frac1) * rise2 * decay2
        
        result[mask] = amp * (component1 + component2)
    
    return result

def model_two_component_cooperative(t, amp_fast, tau_rise_fast, tau_decay_fast, n_fast, 
                                   amp_slow, tau_rise_slow, tau_decay_slow, n_slow, t_peak):
    """Two independent cooperative components (fast + slow)"""
    t = np.asarray(t)
    result = np.zeros_like(t, dtype=float)
    
    mask = t >= t_peak
    if np.any(mask):
        t_shifted = (t[mask] - t_peak) / 1000
        
        # Fast component
        tau_rise_fast_safe = max(tau_rise_fast, 1e-6)
        n_fast_safe = max(n_fast, 0.5)
        normalized_t_fast = t_shifted / tau_rise_fast_safe
        rise_fast = (normalized_t_fast ** n_fast_safe) / (1 + normalized_t_fast ** n_fast_safe)
        decay_fast = np.exp(-t_shifted / max(tau_decay_fast, 1e-6))
        component_fast = amp_fast * rise_fast * decay_fast
        
        # Slow component
        tau_rise_slow_safe = max(tau_rise_slow, 1e-6)
        n_slow_safe = max(n_slow, 0.5)
        normalized_t_slow = t_shifted / tau_rise_slow_safe
        rise_slow = (normalized_t_slow ** n_slow_safe) / (1 + normalized_t_slow ** n_slow_safe)
        decay_slow = np.exp(-t_shifted / max(tau_decay_slow, 1e-6))
        component_slow = amp_slow * rise_slow * decay_slow
        
        result[mask] = component_fast + component_slow
    
    return result

# =============================================================================
# COMPLETE MODEL DICTIONARY (Ordered by Complexity)
# =============================================================================

models_to_test = {
    # 3-parameter models
    'single_exp': {
        'func': model_single_exp_constrained,
        'params': ['amp', 'tau_decay', 't_peak'],
        'bounds': ([0, 0.001, 0], [np.inf, 0.200, 10]),
        'p0_func': lambda y, t: [np.max(y), 0.020, t[np.argmax(y)]],
        'complexity': 3
    },
    'alpha': {
        'func': model_alpha_constrained,
        'params': ['amp', 'tau', 't_peak'],
        'bounds': ([0, 0.001, 0], [np.inf, 0.100, 10]),
        'p0_func': lambda y, t: [np.max(y)*np.e, 0.010, t[np.argmax(y)]],
        'complexity': 3
    },
    
    # 4-parameter models
    'double_exp': {
        'func': model_double_exp_constrained,
        'params': ['amp', 'tau_rise', 'tau_decay', 't_peak'],
        'bounds': ([0, 0.0005, 0.001, 0], [np.inf, 0.010, 0.200, 10]),
        'p0_func': lambda y, t: [np.max(y), 0.002, 0.020, t[np.argmax(y)]],
        'complexity': 4
    },
    'gamma': {
        'func': model_gamma_constrained,
        'params': ['amp', 'n', 'tau', 't_peak'],
        'bounds': ([0, 0.5, 0.001, 0], [np.inf, 8.0, 0.100, 10]),
        'p0_func': lambda y, t: [np.max(y)*3, 2.0, 0.010, t[np.argmax(y)]],
        'complexity': 4
    },
    'bilinear': {
        'func': model_bilinear_constrained,
        'params': ['amp', 't_rise', 't_decay', 't_peak'],
        'bounds': ([0, 0.1, 1, 0], [np.inf, 10, 100, 10]),
        'p0_func': lambda y, t: [np.max(y), 2.0, 20.0, t[np.argmax(y)]],
        'complexity': 4
    },
    
    # 5-parameter models
    'cooperative': {
        'func': model_cooperative_binding,
        'params': ['amp', 'tau_rise', 'tau_decay', 'n_coop', 't_peak'],
        'bounds': ([0, 0.001, 0.005, 0.5, 0], [np.inf, 0.020, 0.200, 5.0, 10]),
        'p0_func': lambda y, t: [np.max(y), 0.005, 0.030, 2.0, t[np.argmax(y)]],
        'complexity': 5
    },
    'binding_kinetics': {
        'func': model_binding_kinetics,
        'params': ['amp', 'kon', 'koff', 'tau_clear', 't_peak'],
        'bounds': ([0, 10, 1, 0.001, 0], [np.inf, 1000, 200, 0.200, 10]),
        'p0_func': lambda y, t: [np.max(y), 200, 50, 0.030, t[np.argmax(y)]],
        'complexity': 5
    },
    
    # 6-parameter models
    'two_component': {
        'func': model_two_component_shared_rise,
        'params': ['amp_fast', 'tau_rise', 'tau_fast', 'amp_slow', 'tau_slow', 't_peak'],
        'bounds': ([0, 0.0005, 0.001, 0, 0.010, 0], [np.inf, 0.010, 0.100, np.inf, 1.000, 10]),
        'p0_func': lambda y, t: [np.max(y)*0.6, 0.002, 0.015, np.max(y)*0.4, 0.080, t[np.argmax(y)]],
        'complexity': 6
    },
    'desensitization': {
        'func': model_desensitization,
        'params': ['amp', 'tau_rise', 'tau_decay', 'tau_recovery', 'desens_factor', 't_peak'],
        'bounds': ([0, 0.001, 0.005, 0.020, 0, 0], [np.inf, 0.010, 0.100, 1.000, 0.8, 10]),
        'p0_func': lambda y, t: [np.max(y), 0.003, 0.020, 0.100, 0.3, t[np.argmax(y)]],
        'complexity': 6
    },
    
    # 7-parameter models
    'coop_plus_linear': {
        'func': model_cooperative_plus_linear,
        'params': ['amp_coop', 'tau_rise_coop', 'tau_decay_coop', 'n_coop', 'amp_linear', 'tau_decay_linear', 't_peak'],
        'bounds': ([0, 0.001, 0.005, 0.5, 0, 0.010, 0], [np.inf, 0.020, 0.200, 5.0, np.inf, 0.500, 10]),
        'p0_func': lambda y, t: [np.max(y)*0.8, 0.005, 0.030, 2.0, np.max(y)*0.2, 0.100, t[np.argmax(y)]],
        'complexity': 7
    },
    'diffusion_clearance': {
        'func': model_diffusion_clearance,
        'params': ['amp', 'tau_diff', 'tau_clear1', 'tau_clear2', 'frac_clear1', 't_peak'],
        'bounds': ([0, 0.001, 0.005, 0.020, 0.1, 0], [np.inf, 0.020, 0.100, 0.500, 0.9, 10]),
        'p0_func': lambda y, t: [np.max(y)*np.e, 0.003, 0.015, 0.080, 0.6, t[np.argmax(y)]],
        'complexity': 7
    },
    
    # 8-parameter models
    'double_cooperative': {
        'func': model_double_cooperative,
        'params': ['amp', 'tau_rise1', 'tau_decay1', 'n1', 'tau_rise2', 'tau_decay2', 'n2', 't_peak'],
        'bounds': ([0, 0.001, 0.005, 0.5, 0.005, 0.020, 0.5, 0], [np.inf, 0.020, 0.100, 5.0, 0.100, 0.500, 5.0, 10]),
        'p0_func': lambda y, t: [np.max(y), 0.003, 0.015, 2.0, 0.010, 0.080, 1.5, t[np.argmax(y)]],
        'complexity': 8
    },
    
    # 9-parameter models (most complex)
    'hetero_coop': {
        'func': model_heterogeneous_cooperative,
        'params': ['amp', 'tau_rise1', 'tau_decay1', 'n1', 'frac1', 'tau_rise2', 'tau_decay2', 'n2', 't_peak'],
        'bounds': ([0, 0.001, 0.005, 0.5, 0.1, 0.005, 0.020, 0.5, 0], [np.inf, 0.020, 0.200, 5.0, 0.9, 0.100, 1.000, 5.0, 10]),
        'p0_func': lambda y, t: [np.max(y), 0.003, 0.020, 2.0, 0.6, 0.010, 0.080, 1.5, t[np.argmax(y)]],
        'complexity': 9
    },
    'two_comp_coop': {
        'func': model_two_component_cooperative,
        'params': ['amp_fast', 'tau_rise_fast', 'tau_decay_fast', 'n_fast', 'amp_slow', 'tau_rise_slow', 'tau_decay_slow', 'n_slow', 't_peak'],
        'bounds': ([0, 0.001, 0.005, 0.5, 0, 0.005, 0.020, 0.5, 0], [np.inf, 0.020, 0.100, 5.0, np.inf, 0.100, 1.000, 5.0, 10]),
        'p0_func': lambda y, t: [np.max(y)*0.6, 0.003, 0.015, 2.0, np.max(y)*0.4, 0.010, 0.080, 1.5, t[np.argmax(y)]],
        'complexity': 9
    }
}

# Print summary organized by complexity
print("=== COMPLETE MODEL LIBRARY (by complexity) ===")
print("For iGluSnFR 2-photon responses - all models have baseline=0, asymptote=0")
print()

for complexity in sorted(set(info['complexity'] for info in models_to_test.values())):
    models_at_level = [(name, info) for name, info in models_to_test.items() if info['complexity'] == complexity]
    print(f"{complexity}-PARAMETER MODELS:")
    for name, info in models_at_level:
        print(f"  {name}: {', '.join(info['params'])}")
    print()

print(f"Total models: {len(models_to_test)}")
print("Complexity range: 3-9 parameters")
print("Recommended progression: Start with 3-4 parameter models, then increase complexity if needed")

In [ ]:
# =============================================================================
# CELL 3C: Model Comparison with Constraints (FIXED - Self-Contained)
# =============================================================================

# Check if we have the necessary data from previous cells
if 'time_analysis' not in globals() or 'y_avg' not in globals():
    print("ERROR: Missing data from Cell 3A. Please run Cell 3A first.")
    print("Required variables: time_analysis, y_avg, avg_noise, traces_array")
else:
    # Set up the fitting region (this should have been in Cell 3A)
    fit_mask = (time_analysis >= 0) & (time_analysis <= 30)
    t_fit = time_analysis[fit_mask]
    y_fit_data = y_avg[fit_mask]
    
    def calculate_constraint_violations(t, y_pred, t_peak):
        """Check how well the model respects physical constraints"""
        
        # Check baseline constraint (before t_peak should be ~0)
        baseline_mask = t < t_peak
        if np.any(baseline_mask):
            baseline_violation = np.mean(np.abs(y_pred[baseline_mask]))
        else:
            baseline_violation = 0
        
        # Check asymptotic constraint (late times should approach 0)
        # Use last 25% of time points
        n_late = max(1, len(t) // 4)
        late_mask = np.zeros(len(t), dtype=bool)
        late_mask[-n_late:] = True
        asymptote_violation = np.mean(np.abs(y_pred[late_mask]))
        
        return baseline_violation, asymptote_violation

    def calculate_model_metrics_constrained(y_data, y_fit, n_params, noise_level, t_fit, y_pred_full, t_full, t_peak):
        """Enhanced metrics including constraint violations"""
        
        # Original metrics
        residuals = y_data - y_fit
        n_points = len(y_data)
        
        mse = np.mean(residuals**2)
        rmse = np.sqrt(mse)
        
        # R-squared
        ss_res = np.sum(residuals**2)
        ss_tot = np.sum((y_data - np.mean(y_data))**2)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        # Adjusted R-squared
        adj_r_squared = 1 - (1 - r_squared) * (n_points - 1) / (n_points - n_params - 1)
        
        # Information criteria
        log_likelihood = -0.5 * n_points * np.log(2 * np.pi * mse) - 0.5 * ss_res / mse
        aic = 2 * n_params - 2 * log_likelihood
        bic = n_params * np.log(n_points) - 2 * log_likelihood
        
        # Noise comparison
        residual_std = np.std(residuals)
        noise_ratio = residual_std / noise_level if noise_level > 0 else np.inf
        
        # Constraint violations
        baseline_violation, asymptote_violation = calculate_constraint_violations(t_full, y_pred_full, t_peak)
        
        # Normality test
        if len(residuals) >= 3:
            try:
                shapiro_stat, shapiro_p = stats.shapiro(residuals)
            except:
                shapiro_stat, shapiro_p = np.nan, np.nan
        else:
            shapiro_stat, shapiro_p = np.nan, np.nan
        
        # Runs test for randomness in residuals
        def runs_test(residuals):
            """Simple runs test for randomness"""
            median_res = np.median(residuals)
            runs = 0
            n1 = n2 = 0
            
            # Count runs
            prev_above = None
            for r in residuals:
                above = r > median_res
                if above != prev_above:
                    runs += 1
                if above:
                    n1 += 1
                else:
                    n2 += 1
                prev_above = above
            
            # Expected runs and standard deviation
            if n1 > 0 and n2 > 0:
                expected_runs = (2 * n1 * n2) / (n1 + n2) + 1
                var_runs = (2 * n1 * n2 * (2 * n1 * n2 - n1 - n2)) / ((n1 + n2)**2 * (n1 + n2 - 1))
                
                if var_runs > 0:
                    z_score = (runs - expected_runs) / np.sqrt(var_runs)
                    p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))
                    return runs, expected_runs, p_value
            
            return runs, np.nan, np.nan
        
        runs, expected_runs, runs_p = runs_test(residuals)
        
        return {
            'mse': mse,
            'rmse': rmse,
            'r_squared': r_squared,
            'adj_r_squared': adj_r_squared,
            'aic': aic,
            'bic': bic,
            'residual_std': residual_std,
            'noise_ratio': noise_ratio,
            'baseline_violation': baseline_violation,
            'asymptote_violation': asymptote_violation,
            'shapiro_stat': shapiro_stat,
            'shapiro_p': shapiro_p,
            'runs': runs,
            'expected_runs': expected_runs,
            'runs_p': runs_p,
            'residuals': residuals
        }

    # Check if models_to_test exists, if not recreate essential ones
    if 'models_to_test' not in globals():
        print("Recreating essential models...")
        
        # Define essential constrained models
        def model_bilinear_constrained(t, amp, t_rise, t_decay, t_peak):
            t = np.asarray(t)
            result = np.zeros_like(t, dtype=float)
            
            rise_mask = (t >= t_peak) & (t <= t_peak + t_rise)
            if np.any(rise_mask):
                t_rel = t[rise_mask] - t_peak
                result[rise_mask] = amp * (t_rel / max(t_rise, 0.1))
            
            decay_mask = t > (t_peak + t_rise)
            if np.any(decay_mask):
                t_rel = (t[decay_mask] - t_peak - t_rise) / 1000
                tau_decay_s = max(t_decay / 1000, 1e-6)
                result[decay_mask] = amp * np.exp(-t_rel / tau_decay_s)
            
            return result

        def model_double_exp_constrained(t, amp, tau_rise, tau_decay, t_peak):
            t = np.asarray(t)
            result = np.zeros_like(t, dtype=float)
            
            mask = t >= t_peak
            if np.any(mask):
                t_shifted = (t[mask] - t_peak) / 1000
                tau_rise = max(tau_rise, 1e-6)
                tau_decay = max(tau_decay, 1e-6)
                
                if tau_decay > tau_rise:
                    t_opt = tau_rise * tau_decay / (tau_decay - tau_rise) * np.log(tau_decay / tau_rise)
                    norm_factor = np.exp(-t_opt / tau_decay) - np.exp(-t_opt / tau_rise)
                    
                    rise_term = np.exp(-t_shifted / tau_rise)
                    decay_term = np.exp(-t_shifted / tau_decay)
                    
                    if norm_factor > 1e-10:
                        result[mask] = amp * (decay_term - rise_term) / norm_factor
                    else:
                        result[mask] = amp * (decay_term - rise_term)
            return result

        # Essential models dictionary
        models_to_test = {
            'bilinear': {
                'func': model_bilinear_constrained,
                'params': ['amp', 't_rise', 't_decay', 't_peak'],
                'bounds': ([0, 0.1, 1, 0], [np.inf, 10, 100, 10]),
                'p0_func': lambda y, t: [np.max(y), 2.0, 20.0, t[np.argmax(y)]]
            },
            'double_exp': {
                'func': model_double_exp_constrained,
                'params': ['amp', 'tau_rise', 'tau_decay', 't_peak'],
                'bounds': ([0, 0.0005, 0.001, 0], [np.inf, 0.010, 0.200, 10]),
                'p0_func': lambda y, t: [np.max(y), 0.002, 0.020, t[np.argmax(y)]]
            }
        }

    # Fit all constrained models
    fit_results = {}

    print("=== CONSTRAINED MODEL FITTING ===")
    print(f"Fitting on {len(t_fit)} points from {t_fit[0]:.1f} to {t_fit[-1]:.1f} ms")
    print(f"Target noise level: {avg_noise:.4f}")
    print()

    for model_name, model_info in models_to_test.items():
        try:
            # Get initial parameters
            p0 = model_info['p0_func'](y_fit_data, t_fit)
            
            # Fit model
            popt, pcov = curve_fit(
                model_info['func'], 
                t_fit, y_fit_data,
                p0=p0,
                bounds=model_info['bounds'],
                maxfev=3000
            )
            
            # Generate predictions
            y_pred_full = model_info['func'](time_analysis, *popt)
            y_pred_fit = y_pred_full[fit_mask]
            
            # Calculate enhanced metrics
            t_peak_fitted = popt[-1]  # t_peak is always last parameter
            metrics = calculate_model_metrics_constrained(
                y_fit_data, y_pred_fit, len(popt), avg_noise, 
                t_fit, y_pred_full, time_analysis, t_peak_fitted
            )
            
            fit_results[model_name] = {
                'params': popt,
                'param_names': model_info['params'],
                'covariance': pcov,
                'y_pred_full': y_pred_full,
                'y_pred_fit': y_pred_fit,
                'metrics': metrics,
                'success': True
            }
            
            # Print detailed summary
            print(f"{model_name.upper()}:")
            for i, (name, val) in enumerate(zip(model_info['params'], popt)):
                if 'tau' in name and 'peak' not in name:
                    print(f"  {name}: {val*1000:.2f} ms")
                elif 't_' in name:
                    print(f"  {name}: {val:.2f} ms")
                else:
                    print(f"  {name}: {val:.3f}")
            print(f"  R² = {metrics['r_squared']:.3f}, AIC = {metrics['aic']:.1f}")
            print(f"  Noise ratio: {metrics['noise_ratio']:.2f}")
            print(f"  Baseline violation: {metrics['baseline_violation']:.4f}")
            print(f"  Asymptote violation: {metrics['asymptote_violation']:.4f}")
            print(f"  Shapiro p: {metrics['shapiro_p']:.3f}, Runs p: {metrics['runs_p']:.3f}")
            print()
            
        except Exception as e:
            fit_results[model_name] = {
                'success': False,
                'error': str(e),
                'metrics': {'aic': np.inf, 'bic': np.inf, 'noise_ratio': np.inf, 
                           'baseline_violation': np.inf, 'asymptote_violation': np.inf}
            }
            print(f"{model_name.upper()}: FAILED - {e}")
            print()

    # Enhanced model ranking
    successful_models = {k: v for k, v in fit_results.items() if v['success']}

    if successful_models:
        print("=== ENHANCED MODEL RANKING ===")
        
        # Multiple ranking criteria
        rankings = {}
        
        # By constraint violations (lower is better)
        rankings['Constraints'] = sorted(successful_models.keys(), 
                                       key=lambda x: successful_models[x]['metrics']['baseline_violation'] + 
                                                    successful_models[x]['metrics']['asymptote_violation'])
        
        # By AIC (lower is better)
        rankings['AIC'] = sorted(successful_models.keys(), 
                                key=lambda x: successful_models[x]['metrics']['aic'])
        
        # By noise ratio (closer to 1 is better)
        rankings['Noise_Ratio'] = sorted(successful_models.keys(), 
                                       key=lambda x: abs(successful_models[x]['metrics']['noise_ratio'] - 1))
        
        for criterion, ranked_models in rankings.items():
            print(f"{criterion}:")
            for i, model in enumerate(ranked_models[:3]):
                metrics = successful_models[model]['metrics']
                if criterion == 'Constraints':
                    val = metrics['baseline_violation'] + metrics['asymptote_violation']
                elif criterion == 'AIC':
                    val = metrics['aic']
                elif criterion == 'Noise_Ratio':
                    val = metrics['noise_ratio']
                print(f"  {i+1}. {model}: {val:.3f}")
            print()
        
        # Composite scoring with constraint penalties
        best_model = None
        best_score = np.inf
        
        for model_name in successful_models:
            metrics = successful_models[model_name]['metrics']
            
            # Composite score with heavy penalty for constraint violations
            constraint_penalty = (metrics['baseline_violation'] + metrics['asymptote_violation']) * 1000
            aic_norm = metrics['aic'] / 100
            noise_dev = abs(metrics['noise_ratio'] - 1) * 10
            randomness_penalty = (1 - metrics['runs_p']) * 5 if np.isfinite(metrics['runs_p']) else 5
            
            score = aic_norm + noise_dev + constraint_penalty + randomness_penalty
            
            if score < best_score:
                best_score = score
                best_model = model_name
        
        print(f"BEST CONSTRAINED MODEL: {best_model} (composite score: {best_score:.3f})")
        
    else:
        print("No models fitted successfully!")
        best_model = None

In [ ]:
# =============================================================================
# CELL 3D: Individual Model Analysis (Updated)
# =============================================================================

if len(successful_models) > 0:
    n_models = len(successful_models)
    
    # Create a large figure with subplots for each model
    fig = plt.figure(figsize=(18, 4 * n_models))
    
    model_names = list(successful_models.keys())
    
    for i, model_name in enumerate(model_names):
        result = successful_models[model_name]
        metrics = result['metrics']
        
        # Create 3 subplots for this model: fit, residuals, residual histogram
        ax1 = plt.subplot(n_models, 3, 3*i + 1)
        ax2 = plt.subplot(n_models, 3, 3*i + 2)
        ax3 = plt.subplot(n_models, 3, 3*i + 3)
        
        # Panel 1: Fit
        ax1.plot(time_analysis, y_avg, 'b-', linewidth=2, label='Data', alpha=0.8)
        ax1.plot(time_analysis, result['y_pred_full'], 'r--', linewidth=2, 
                label=f'{model_name} fit', alpha=0.9)
        ax1.axvline(0, color='k', linestyle=':', alpha=0.5)
        ax1.axhline(0, color='k', linestyle='-', alpha=0.3)
        ax1.set_xlabel('Time (ms)')
        ax1.set_ylabel('Signal')
        ax1.set_title(f'{model_name.upper()} - R²={metrics["r_squared"]:.3f}')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Add parameter text
        param_text = []
        for name, val in zip(result['param_names'], result['params']):
            if 'tau' in name and 'peak' not in name:
                param_text.append(f'{name}={val*1000:.1f}ms')
            elif 't_' in name:
                param_text.append(f'{name}={val:.1f}ms')
            else:
                param_text.append(f'{name}={val:.2f}')
        
        ax1.text(0.02, 0.98, '\n'.join(param_text), transform=ax1.transAxes,
                verticalalignment='top', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        # Panel 2: Residuals
        residuals = metrics['residuals']
        ax2.plot(t_fit, residuals, 'ko-', markersize=3, linewidth=1)
        ax2.axhline(0, color='r', linestyle='--', alpha=0.7)
        ax2.axhline(avg_noise, color='g', linestyle='--', alpha=0.7, 
                   label=f'±{avg_noise:.3f}')
        ax2.axhline(-avg_noise, color='g', linestyle='--', alpha=0.7)
        ax2.set_xlabel('Time (ms)')
        ax2.set_ylabel('Residuals')
        ax2.set_title(f'Residuals - Noise Ratio: {metrics["noise_ratio"]:.2f}')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Panel 3: Residual histogram
        ax3.hist(residuals, bins=min(15, len(residuals)//2), density=True, 
                alpha=0.7, color='skyblue', edgecolor='black')
        
        # Overlay normal distribution
        if len(residuals) > 2:
            x_norm = np.linspace(residuals.min(), residuals.max(), 100)
            normal_fit = stats.norm(np.mean(residuals), np.std(residuals))
            ax3.plot(x_norm, normal_fit.pdf(x_norm), 'r-', linewidth=2, label='Normal')
            
            # Add Shapiro p-value
            ax3.text(0.02, 0.98, f'Shapiro p={metrics["shapiro_p"]:.3f}\nRuns p={metrics["runs_p"]:.3f}', 
                    transform=ax3.transAxes, verticalalignment='top', fontsize=8,
                    bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
        
        ax3.set_xlabel('Residual Value')
        ax3.set_ylabel('Density')
        ax3.set_title('Residual Distribution')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("=== INDIVIDUAL MODEL ANALYSIS ===")
    for model_name in model_names:
        metrics = successful_models[model_name]['metrics']
        print(f"\n{model_name.upper()}:")
        print(f"  R² = {metrics['r_squared']:.4f}")
        print(f"  AIC = {metrics['aic']:.1f}")
        print(f"  Noise ratio = {metrics['noise_ratio']:.2f} (ideal=1.0)")
        print(f"  Baseline violation = {metrics['baseline_violation']:.4f}")
        print(f"  Asymptote violation = {metrics['asymptote_violation']:.4f}")
        print(f"  Residuals normal? p = {metrics['shapiro_p']:.3f} (>0.05 is good)")
        print(f"  Residuals random? p = {metrics['runs_p']:.3f} (>0.05 is good)")

else:
    print("No successful models to analyze!")

In [ ]:
# =============================================================================
# CELL 3E: Summary Comparison Panel
# =============================================================================

if len(successful_models) > 0:
    # Create comprehensive summary figure
    fig = plt.figure(figsize=(20, 12))
    
    # Panel 1: All model overlays
    ax1 = plt.subplot(2, 4, 1)
    ax1.plot(time_analysis, y_avg, 'k-', linewidth=3, alpha=0.8, label='Data')
    
    colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown']
    linestyles = ['-', '--', '-.', ':', '-', '--']
    
    for i, (model_name, result) in enumerate(successful_models.items()):
        if i < len(colors):
            alpha = 1.0 if model_name == best_model else 0.7
            linewidth = 3 if model_name == best_model else 2
            ax1.plot(time_analysis, result['y_pred_full'], 
                    color=colors[i], linestyle=linestyles[i], 
                    linewidth=linewidth, alpha=alpha, label=model_name)
    
    ax1.axvline(0, color='k', linestyle=':', alpha=0.5)
    ax1.set_xlabel('Time (ms)')
    ax1.set_ylabel('Signal')
    ax1.set_title('Model Comparison')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: R² comparison
    ax2 = plt.subplot(2, 4, 2)
    model_names = list(successful_models.keys())
    r_squared_vals = [successful_models[m]['metrics']['r_squared'] for m in model_names]
    
    bars = ax2.bar(range(len(model_names)), r_squared_vals, 
                   color=['red' if m == best_model else 'lightblue' for m in model_names])
    ax2.set_xticks(range(len(model_names)))
    ax2.set_xticklabels(model_names, rotation=45, ha='right')
    ax2.set_ylabel('R²')
    ax2.set_title('Model R² Comparison')
    ax2.grid(True, alpha=0.3)
    
    # Add values on bars
    for i, v in enumerate(r_squared_vals):
        ax2.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    
    # Panel 3: AIC comparison (lower is better)
    ax3 = plt.subplot(2, 4, 3)
    aic_vals = [successful_models[m]['metrics']['aic'] for m in model_names]
    
    bars = ax3.bar(range(len(model_names)), aic_vals,
                   color=['red' if m == best_model else 'lightcoral' for m in model_names])
    ax3.set_xticks(range(len(model_names)))
    ax3.set_xticklabels(model_names, rotation=45, ha='right')
    ax3.set_ylabel('AIC')
    ax3.set_title('AIC Comparison (lower=better)')
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: Noise ratio (closer to 1 is better)
    ax4 = plt.subplot(2, 4, 4)
    noise_ratios = [successful_models[m]['metrics']['noise_ratio'] for m in model_names]
    
    bars = ax4.bar(range(len(model_names)), noise_ratios,
                   color=['red' if m == best_model else 'lightgreen' for m in model_names])
    ax4.axhline(1.0, color='darkgreen', linestyle='--', linewidth=2, label='Ideal (1.0)')
    ax4.set_xticks(range(len(model_names)))
    ax4.set_xticklabels(model_names, rotation=45, ha='right')
    ax4.set_ylabel('Noise Ratio')
    ax4.set_title('Noise Ratio (closer to 1=better)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # Panel 5: Constraint violations
    ax5 = plt.subplot(2, 4, 5)
    baseline_violations = [successful_models[m]['metrics']['baseline_violation'] for m in model_names]
    asymptote_violations = [successful_models[m]['metrics']['asymptote_violation'] for m in model_names]
    
    x = np.arange(len(model_names))
    width = 0.35
    
    bars1 = ax5.bar(x - width/2, baseline_violations, width, label='Baseline', alpha=0.7)
    bars2 = ax5.bar(x + width/2, asymptote_violations, width, label='Asymptote', alpha=0.7)
    
    ax5.set_xticks(x)
    ax5.set_xticklabels(model_names, rotation=45, ha='right')
    ax5.set_ylabel('Constraint Violation')
    ax5.set_title('Constraint Violations (lower=better)')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # Panel 6: Residual statistics
    ax6 = plt.subplot(2, 4, 6)
    shapiro_ps = [successful_models[m]['metrics']['shapiro_p'] for m in model_names]
    runs_ps = [successful_models[m]['metrics']['runs_p'] for m in model_names]
    
    x = np.arange(len(model_names))
    width = 0.35
    
    bars1 = ax6.bar(x - width/2, shapiro_ps, width, label='Shapiro p', alpha=0.7)
    bars2 = ax6.bar(x + width/2, runs_ps, width, label='Runs p', alpha=0.7)
    ax6.axhline(0.05, color='red', linestyle='--', alpha=0.7, label='p=0.05')
    
    ax6.set_xticks(x)
    ax6.set_xticklabels(model_names, rotation=45, ha='right')
    ax6.set_ylabel('p-value')
    ax6.set_title('Residual Tests (>0.05=good)')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    # Panel 7: Best model residuals vs time
    ax7 = plt.subplot(2, 4, 7)
    if best_model:
        best_residuals = successful_models[best_model]['metrics']['residuals']
        ax7.plot(t_fit, best_residuals, 'ko-', markersize=4, linewidth=1)
        ax7.axhline(0, color='r', linestyle='--', alpha=0.7)
        ax7.axhline(avg_noise, color='g', linestyle='--', alpha=0.7)
        ax7.axhline(-avg_noise, color='g', linestyle='--', alpha=0.7)
        ax7.set_xlabel('Time (ms)')
        ax7.set_ylabel('Residuals')
        ax7.set_title(f'Best Model ({best_model}) Residuals')
        ax7.grid(True, alpha=0.3)
    
    # Panel 8: Performance scatter plot
    ax8 = plt.subplot(2, 4, 8)
    aic_vals = [successful_models[m]['metrics']['aic'] for m in model_names]
    noise_ratios = [successful_models[m]['metrics']['noise_ratio'] for m in model_names]
    
    colors_scatter = ['red' if m == best_model else 'blue' for m in model_names]
    sizes = [100 if m == best_model else 60 for m in model_names]
    
    scatter = ax8.scatter(aic_vals, noise_ratios, c=colors_scatter, s=sizes, alpha=0.7)
    
    for i, name in enumerate(model_names):
        ax8.annotate(name, (aic_vals[i], noise_ratios[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    ax8.axhline(1.0, color='g', linestyle='--', alpha=0.7, label='Ideal noise ratio')
    ax8.set_xlabel('AIC (lower is better)')
    ax8.set_ylabel('Noise Ratio (closer to 1 is better)')
    ax8.set_title('Model Performance Space')
    ax8.legend()
    ax8.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary table
    print("\n=== SUMMARY TABLE ===")
    print(f"{'Model':<12} {'R²':<6} {'AIC':<6} {'Noise':<6} {'Base':<6} {'Asym':<6} {'Shap':<6} {'Runs':<6}")
    print("-" * 70)
    
    for model_name in model_names:
        m = successful_models[model_name]['metrics']
        marker = ">>> " if model_name == best_model else "    "
        print(f"{marker}{model_name:<8} {m['r_squared']:.3f}  {m['aic']:.0f}   {m['noise_ratio']:.2f}   {m['baseline_violation']:.3f}  {m['asymptote_violation']:.3f}  {m['shapiro_p']:.3f}  {m['runs_p']:.3f}")

else:
    print("No successful models to compare!")

In [ ]:
# =============================================================================
# CELL 3F: Export Best Model for Individual Trial Analysis (FIXED)
# =============================================================================

# Export the best model information for use in next cells
if best_model and best_model in successful_models:
    
    # Get the best model details
    best_result = successful_models[best_model]
    best_function = models_to_test[best_model]['func']  # This IS the constrained version
    best_params = best_result['params']
    best_param_names = best_result['param_names']
    
    print(f"DEBUG: Best model {best_model} has parameters: {best_param_names}")
    
    # The models_to_test should already contain constrained versions without baseline
    # If there's still a baseline parameter, we need to identify the issue
    if 'baseline' in best_param_names:
        print("ERROR: Baseline parameter found - this should not happen with constrained models")
        print("Please check that Cell 3B was run with constrained models")
        BEST_MODEL = None
    else:
        # Create a convenient model object
        class BestModelExporter:
            def __init__(self, name, func, params, param_names):
                self.name = name
                self.func = func
                self.params = params  # Best fit parameters from average
                self.param_names = param_names
                
            def predict(self, t, params=None):
                """Predict using this model"""
                if params is None:
                    params = self.params
                return self.func(t, *params)
                
            def get_varied_bounds(self, jitter_ms=2.0, tau_factor=10.0):
                """Get bounds for individual trial fitting with parameter variations"""
                bounds_lower = []
                bounds_upper = []
                
                for i, (name, val) in enumerate(zip(self.param_names, self.params)):
                    if name == 'amp':
                        # Amplitude can vary widely but must be positive
                        bounds_lower.append(0)
                        bounds_upper.append(max(val * 5, 200))  # More reasonable upper bound
                    elif 't_peak' in name:
                        # Peak time can jitter by ±jitter_ms, but must be >= 0
                        bounds_lower.append(0)  # Force minimum to be 0
                        bounds_upper.append(max(val + jitter_ms, jitter_ms))
                    elif 'tau' in name:
                        # Tau parameters can vary within 1 order of magnitude
                        bounds_lower.append(val / tau_factor)
                        bounds_upper.append(val * tau_factor)
                    elif 't_rise' in name or 't_decay' in name:
                        # Time constants for bilinear model
                        bounds_lower.append(val / tau_factor)
                        bounds_upper.append(val * tau_factor)
                    elif name == 'n':
                        # Shape parameter for gamma
                        bounds_lower.append(max(0.5, val / 2))
                        bounds_upper.append(val * 2)
                    else:
                        # Default: allow variation but keep positive
                        bounds_lower.append(max(0, val * 0.1))
                        bounds_upper.append(val * 3)
                        
                return bounds_lower, bounds_upper
                
            def get_initial_guess(self, data_peak_amp=None, data_peak_time=None):
                """Get initial guess for fitting, ensuring it's within bounds"""
                p0 = list(self.params.copy())
                
                # Get bounds to ensure initial guess is valid
                bounds_lower, bounds_upper = self.get_varied_bounds()
                
                if data_peak_amp is not None and data_peak_amp > 0:
                    # Use actual data peak amplitude if available
                    amp_idx = [i for i, name in enumerate(self.param_names) if name == 'amp']
                    if amp_idx:
                        p0[amp_idx[0]] = max(bounds_lower[amp_idx[0]], 
                                            min(bounds_upper[amp_idx[0]], data_peak_amp))
                        
                if data_peak_time is not None and np.isfinite(data_peak_time):
                    # Use actual data peak time if available
                    peak_idx = [i for i, name in enumerate(self.param_names) if 't_peak' in name]
                    if peak_idx:
                        p0[peak_idx[0]] = max(bounds_lower[peak_idx[0]], 
                                             min(bounds_upper[peak_idx[0]], data_peak_time))
                
                # Ensure all parameters are within bounds
                for i in range(len(p0)):
                    p0[i] = max(bounds_lower[i], min(bounds_upper[i], p0[i]))
                        
                return p0
        
        # Create the exported model
        BEST_MODEL = BestModelExporter(
            name=best_model,
            func=best_function,
            params=best_params,
            param_names=best_param_names
        )
        
        print(f"=== BEST MODEL EXPORTED: {best_model.upper()} ===")
        print("Parameters from average fit:")
        for name, val in zip(best_param_names, best_params):
            if 'tau' in name and 'peak' not in name:
                print(f"  {name}: {val*1000:.2f} ms")
            elif 't_' in name:
                print(f"  {name}: {val:.2f} ms")
            else:
                print(f"  {name}: {val:.3f}")
        
        print(f"\nModel ready for individual trial fitting")
        print(f"Parameters: {BEST_MODEL.param_names}")
        
        # Quick test of the model
        try:
            test_prediction = BEST_MODEL.predict(np.array([0, 5, 10, 20]))
            print(f"✓ Model test successful")
        except Exception as e:
            print(f"✗ Model test failed: {e}")
            BEST_MODEL = None

else:
    print("ERROR: No best model found - cannot proceed to individual trial analysis")
    BEST_MODEL = None

## 4 : Apply model to individual events

In [ ]:
# =============================================================================
# CELL 4A: Individual Trial Fitting Setup (FIXED)
# =============================================================================

if BEST_MODEL is None:
    print("ERROR: No best model available. Run previous cells first.")
else:
    from scipy.optimize import curve_fit
    import warnings
    warnings.filterwarnings('ignore', category=RuntimeWarning)
    
    # Parameters for individual fitting
    PEAK_JITTER_MS = 2.0      # Allow ±2ms peak time variation
    TAU_VARIATION_FACTOR = 10.0  # Allow 1 order of magnitude variation in tau
    
    print(f"=== INDIVIDUAL TRIAL FITTING SETUP ===")
    print(f"Model: {BEST_MODEL.name} (constrained)")
    print(f"Peak jitter allowed: ±{PEAK_JITTER_MS} ms")
    print(f"Tau variation factor: {TAU_VARIATION_FACTOR}x")
    print(f"Trials to fit: {len(traces_array)}")
    print(f"Parameters: {BEST_MODEL.param_names}")
    
    # Get bounds for individual fitting
    bounds_lower, bounds_upper = BEST_MODEL.get_varied_bounds(
        jitter_ms=PEAK_JITTER_MS, 
        tau_factor=TAU_VARIATION_FACTOR
    )
    
    print("\nParameter bounds for individual fitting:")
    for i, name in enumerate(BEST_MODEL.param_names):
        if 'tau' in name and 'peak' not in name:
            print(f"  {name}: [{bounds_lower[i]*1000:.2f}, {bounds_upper[i]*1000:.2f}] ms")
        elif 't_' in name:
            print(f"  {name}: [{bounds_lower[i]:.2f}, {bounds_upper[i]:.2f}] ms")
        else:
            print(f"  {name}: [{bounds_lower[i]:.4f}, {bounds_upper[i]:.4f}]")
    
    # Test the bounds with average parameters
    print(f"\nTesting bounds with average parameters:")
    test_p0 = BEST_MODEL.get_initial_guess()
    print(f"Initial guess: {test_p0}")
    
    bounds_ok = True
    for i, (p, lower, upper) in enumerate(zip(test_p0, bounds_lower, bounds_upper)):
        if not (lower <= p <= upper):
            print(f"  ERROR: Parameter {i} ({BEST_MODEL.param_names[i]}) = {p} not in [{lower}, {upper}]")
            bounds_ok = False
        else:
            print(f"  ✓ {BEST_MODEL.param_names[i]}: {p:.4f} ∈ [{lower:.4f}, {upper:.4f}]")
    
    if not bounds_ok:
        print("ERROR: Initial guess outside bounds - fixing...")
        # This shouldn't happen with the fixed get_initial_guess method
        BEST_MODEL = None
    else:
        print("✓ Bounds check passed")
        
        # Storage for individual trial results
        individual_fits = {
            'success': [],
            'params': [],
            'residuals': [],
            'metrics': [],
            'predictions': [],
            'trial_indices': []
        }
        
        # Fit each individual trace
        print(f"\nFitting {len(traces_array)} individual trials...")
        
        for trial_idx, trace in enumerate(traces_array):
            try:
                # Find peak in this individual trace for better initial guess
                # Use the fitting region only
                fit_mask_trial = (time_analysis >= 0) & (time_analysis <= 30)
                trace_fit_region = trace[fit_mask_trial]
                time_fit_region = time_analysis[fit_mask_trial]
                
                if np.all(np.isnan(trace_fit_region)) or len(trace_fit_region) < 5:
                    raise ValueError("Insufficient valid data points in fitting region")
                
                # Find peak in fitting region
                valid_indices = np.isfinite(trace_fit_region)
                if not np.any(valid_indices):
                    raise ValueError("No finite values in trace")
                    
                trace_clean = trace_fit_region[valid_indices]
                time_clean = time_fit_region[valid_indices]
                
                if len(trace_clean) < 5:
                    raise ValueError("Insufficient valid data points after cleaning")
                
                peak_idx = np.argmax(trace_clean)
                trace_peak_time = time_clean[peak_idx]
                trace_peak_amp = trace_clean[peak_idx]
                
                # Get initial guess based on this trace's characteristics
                p0 = BEST_MODEL.get_initial_guess(
                    data_peak_amp=trace_peak_amp,
                    data_peak_time=trace_peak_time
                )
                
                # Perform the fit on clean data
                popt, pcov = curve_fit(
                    BEST_MODEL.func,
                    time_clean, trace_clean,
                    p0=p0,
                    bounds=(bounds_lower, bounds_upper),
                    maxfev=2000
                )
                
                # Generate prediction for full time range
                y_pred_full = BEST_MODEL.func(time_analysis, *popt)
                y_pred_clean = BEST_MODEL.func(time_clean, *popt)
                
                # Calculate residuals and basic metrics
                residuals = trace_clean - y_pred_clean
                r_squared = 1 - np.sum(residuals**2) / np.sum((trace_clean - np.mean(trace_clean))**2)
                rmse = np.sqrt(np.mean(residuals**2))
                
                # Store results
                individual_fits['success'].append(True)
                individual_fits['params'].append(popt)
                individual_fits['residuals'].append(residuals)
                individual_fits['predictions'].append(y_pred_full)
                individual_fits['trial_indices'].append(trial_idx)
                individual_fits['metrics'].append({
                    'r_squared': r_squared,
                    'rmse': rmse,
                    'residual_std': np.std(residuals)
                })
                
                if (trial_idx + 1) % 5 == 0:
                    print(f"  Completed {trial_idx + 1}/{len(traces_array)} trials")
                    
            except Exception as e:
                print(f"  Trial {trial_idx + 1} failed: {str(e)[:70]}...")
                individual_fits['success'].append(False)
                individual_fits['params'].append(None)
                individual_fits['residuals'].append(None)
                individual_fits['predictions'].append(None)
                individual_fits['trial_indices'].append(trial_idx)
                individual_fits['metrics'].append(None)
        
        # Summary statistics
        n_successful = sum(individual_fits['success'])
        n_failed = len(individual_fits['success']) - n_successful
        
        print(f"\n=== INDIVIDUAL FITTING RESULTS ===")
        print(f"Successful fits: {n_successful}/{len(traces_array)} ({100*n_successful/len(traces_array):.1f}%)")
        print(f"Failed fits: {n_failed}")
        
        if n_successful > 0:
            # Get successful results only
            successful_params = [p for p, s in zip(individual_fits['params'], individual_fits['success']) if s and p is not None]
            successful_residuals = [r for r, s in zip(individual_fits['residuals'], individual_fits['success']) if s and r is not None]
            successful_metrics = [m for m, s in zip(individual_fits['metrics'], individual_fits['success']) if s and m is not None]
            
            # Parameter statistics
            if successful_params:
                params_array = np.array(successful_params)
                print(f"\nParameter statistics across {len(successful_params)} successful fits:")
                print(f"{'Parameter':<12} {'Mean':<8} {'Std':<8} {'Min':<8} {'Max':<8}")
                print("-" * 50)
                
                for i, name in enumerate(BEST_MODEL.param_names):
                    values = params_array[:, i]
                    if 'tau' in name and 'peak' not in name:
                        print(f"{name:<12} {np.mean(values)*1000:.2f}ms  {np.std(values)*1000:.2f}ms  {np.min(values)*1000:.2f}ms  {np.max(values)*1000:.2f}ms")
                    elif 't_' in name:
                        print(f"{name:<12} {np.mean(values):.2f}ms  {np.std(values):.2f}ms  {np.min(values):.2f}ms  {np.max(values):.2f}ms")
                    else:
                        print(f"{name:<12} {np.mean(values):.3f}    {np.std(values):.3f}    {np.min(values):.3f}    {np.max(values):.3f}")
        
        else:
            print("No successful individual fits - check data quality or model constraints")

In [ ]:
# =============================================================================
# CELL 4B: Individual Trial Residual Analysis
# =============================================================================

if n_successful > 0:
    # Create visualization of individual trial fits and residuals
    
    # Select a subset of trials to show (max 12 for readability)
    max_trials_to_show = 12
    successful_indices = [i for i, s in enumerate(individual_fits['success']) if s]
    
    if len(successful_indices) > max_trials_to_show:
        # Select evenly spaced trials
        step = len(successful_indices) // max_trials_to_show
        show_indices = successful_indices[::step][:max_trials_to_show]
    else:
        show_indices = successful_indices
    
    # Create figure for individual trial analysis
    n_show = len(show_indices)
    n_cols = 4
    n_rows = (n_show + n_cols - 1) // n_cols
    
    fig = plt.figure(figsize=(16, 4 * n_rows))
    
    print(f"=== INDIVIDUAL TRIAL FITS ===")
    print(f"Showing {n_show} representative trials out of {n_successful} successful fits")
    
    for plot_idx, trial_idx in enumerate(show_indices):
        fit_idx = trial_idx  # Index in the individual_fits arrays
        
        # Get the data for this trial
        trace = traces_array[trial_idx]
        params = individual_fits['params'][fit_idx]
        prediction = individual_fits['predictions'][fit_idx]
        residuals = individual_fits['residuals'][fit_idx]
        metrics = individual_fits['metrics'][fit_idx]
        
        if params is None or prediction is None:
            continue
            
        # Create subplot
        ax = plt.subplot(n_rows, n_cols, plot_idx + 1)
        
        # Plot data and fit
        ax.plot(time_analysis, trace, 'b-', linewidth=1.5, alpha=0.7, label='Data')
        ax.plot(time_analysis, prediction, 'r--', linewidth=2, label='Fit')
        ax.axvline(0, color='k', linestyle=':', alpha=0.5)
        ax.axhline(0, color='k', linestyle='-', alpha=0.3)
        
        # Add fit quality info
        ax.text(0.02, 0.98, f'Trial {trial_idx+1}\nR²={metrics["r_squared"]:.3f}', 
                transform=ax.transAxes, verticalalignment='top', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        ax.set_xlabel('Time (ms)')
        ax.set_ylabel('Signal')
        ax.set_title(f'Trial {trial_idx+1}')
        if plot_idx == 0:
            ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Set consistent axis limits
        ax.set_xlim(-5, 50)
    
    plt.tight_layout()
    plt.show()
    
    # Plot residuals per trial
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Panel 1: Residuals vs time for all successful trials
    colors = plt.cm.viridis(np.linspace(0, 1, len(successful_residuals)))
    
    for i, (residuals, color) in enumerate(zip(successful_residuals, colors)):
        if residuals is not None and len(residuals) > 0:
            # Create time axis for these residuals (they're from the fit region)
            fit_mask_residuals = (time_analysis >= 0) & (time_analysis <= 30)
            t_residuals = time_analysis[fit_mask_residuals]
            
            # Handle case where residuals might be shorter due to NaN removal
            if len(residuals) <= len(t_residuals):
                t_plot = t_residuals[:len(residuals)]
                ax1.plot(t_plot, residuals, color=color, alpha=0.3, linewidth=0.8)
    
    ax1.axhline(0, color='red', linestyle='--', linewidth=1)
    ax1.axhline(avg_noise, color='green', linestyle='--', alpha=0.7, label=f'±{avg_noise:.3f}')
    ax1.axhline(-avg_noise, color='green', linestyle='--', alpha=0.7)
    ax1.set_xlabel('Time (ms)')
    ax1.set_ylabel('Residuals')
    ax1.set_title(f'Residuals vs Time ({n_successful} trials)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: R² distribution
    r_squared_values = [m['r_squared'] for m in successful_metrics if m is not None]
    
    ax2.hist(r_squared_values, bins=min(15, len(r_squared_values)//2), 
             alpha=0.7, color='skyblue', edgecolor='black')
    ax2.axvline(np.mean(r_squared_values), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {np.mean(r_squared_values):.3f}')
    ax2.set_xlabel('R²')
    ax2.set_ylabel('Number of Trials')
    ax2.set_title('R² Distribution Across Trials')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"R² statistics: Mean={np.mean(r_squared_values):.3f}, Std={np.std(r_squared_values):.3f}")
    print(f"R² range: {np.min(r_squared_values):.3f} to {np.max(r_squared_values):.3f}")

else:
    print("No successful fits to analyze")

In [ ]:
# =============================================================================
# CELL 4C: Aggregated Residual Distribution Analysis
# =============================================================================

if n_successful > 0:
    # Combine all residuals from successful trials
    all_residuals = []
    all_trial_stds = []
    
    for residuals, metrics in zip(successful_residuals, successful_metrics):
        if residuals is not None and metrics is not None:
            all_residuals.extend(residuals)
            all_trial_stds.append(metrics['residual_std'])
    
    all_residuals = np.array(all_residuals)
    
    print(f"=== AGGREGATED RESIDUAL ANALYSIS ===")
    print(f"Total residual points: {len(all_residuals)}")
    print(f"From {n_successful} successful trial fits")
    
    # Create comprehensive residual analysis figure
    fig = plt.figure(figsize=(16, 10))
    
    # Panel 1: Aggregated residual histogram with normality test
    ax1 = plt.subplot(2, 3, 1)
    
    n_bins = min(50, len(all_residuals)//20)
    counts, bins, patches = ax1.hist(all_residuals, bins=n_bins, density=True, 
                                    alpha=0.7, color='lightblue', edgecolor='black')
    
    # Overlay normal distribution
    x_norm = np.linspace(all_residuals.min(), all_residuals.max(), 200)
    normal_fit = stats.norm(np.mean(all_residuals), np.std(all_residuals))
    ax1.plot(x_norm, normal_fit.pdf(x_norm), 'r-', linewidth=2, label='Normal fit')
    
    # Add noise level reference
    ax1.axvline(avg_noise, color='green', linestyle='--', alpha=0.7, label=f'Target noise: ±{avg_noise:.3f}')
    ax1.axvline(-avg_noise, color='green', linestyle='--', alpha=0.7)
    
    # Normality test
    shapiro_stat, shapiro_p = stats.shapiro(all_residuals) if len(all_residuals) <= 5000 else (np.nan, np.nan)
    
    ax1.set_xlabel('Residual Value')
    ax1.set_ylabel('Density')
    ax1.set_title(f'Aggregated Residuals (n={len(all_residuals)})\nShapiro p={shapiro_p:.4f}')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: Q-Q plot for normality assessment
    ax2 = plt.subplot(2, 3, 2)
    
    from scipy import stats
    stats.probplot(all_residuals, dist="norm", plot=ax2)
    ax2.set_title('Q-Q Plot vs Normal Distribution')
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: Residual std per trial
    ax3 = plt.subplot(2, 3, 3)
    
    trial_numbers = range(1, len(all_trial_stds) + 1)
    ax3.scatter(trial_numbers, all_trial_stds, alpha=0.6, s=30)
    ax3.axhline(avg_noise, color='green', linestyle='--', linewidth=2, 
                label=f'Target noise: {avg_noise:.3f}')
    ax3.axhline(np.mean(all_trial_stds), color='red', linestyle='--', linewidth=2,
                label=f'Mean residual std: {np.mean(all_trial_stds):.3f}')
    
    ax3.set_xlabel('Trial Number')
    ax3.set_ylabel('Residual Standard Deviation')
    ax3.set_title('Residual Std per Trial')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: Comparison with baseline noise
    ax4 = plt.subplot(2, 3, 4)
    
    # Create comparison histogram
    residual_std = np.std(all_residuals)
    
    categories = ['Baseline\nNoise', 'Model\nResiduals']
    values = [avg_noise, residual_std]
    colors = ['green', 'blue']
    
    bars = ax4.bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
    
    # Add ratio text
    ratio = residual_std / avg_noise if avg_noise > 0 else np.inf
    ax4.text(0.5, max(values) * 0.8, f'Ratio: {ratio:.2f}', 
             ha='center', va='center', fontsize=12, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
    
    ax4.set_ylabel('Standard Deviation')
    ax4.set_title('Noise vs Residual Comparison')
    ax4.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + max(values)*0.02,
                f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
    
    # Panel 5: Autocorrelation of residuals
    ax5 = plt.subplot(2, 3, 5)
    
    # Simple autocorrelation calculation
    def autocorrelation(x, max_lag=20):
        n = len(x)
        x = x - np.mean(x)
        autocorr = np.correlate(x, x, mode='full')
        autocorr = autocorr[n-1:]
        autocorr = autocorr / autocorr[0]  # Normalize
        return autocorr[:max_lag+1]
    
    if len(all_residuals) > 50:  # Only if we have enough data
        lags = range(21)
        autocorr = autocorrelation(all_residuals, max_lag=20)
        
        ax5.plot(lags, autocorr, 'bo-', markersize=4)
        ax5.axhline(0, color='red', linestyle='--', alpha=0.7)
        ax5.axhline(0.05, color='green', linestyle='--', alpha=0.5, label='±0.05')
        ax5.axhline(-0.05, color='green', linestyle='--', alpha=0.5)
        
        ax5.set_xlabel('Lag')
        ax5.set_ylabel('Autocorrelation')
        ax5.set_title('Residual Autocorrelation')
        ax5.legend()
        ax5.grid(True, alpha=0.3)
    else:
        ax5.text(0.5, 0.5, 'Insufficient data\nfor autocorrelation', 
                ha='center', va='center', transform=ax5.transAxes)
        ax5.set_title('Residual Autocorrelation')
    
    # Panel 6: Summary statistics
    ax6 = plt.subplot(2, 3, 6)
    ax6.axis('off')
    
    # Calculate additional statistics
    residual_mean = np.mean(all_residuals)
    residual_median = np.median(all_residuals)
    residual_skew = stats.skew(all_residuals)
    residual_kurtosis = stats.kurtosis(all_residuals)
    
    # Create summary text
    summary_text = f"""RESIDUAL SUMMARY STATISTICS
    
Total residual points: {len(all_residuals)}
Successful trials: {n_successful}/{len(traces_array)}

DISTRIBUTION:
Mean: {residual_mean:.4f}
Median: {residual_median:.4f}
Std: {residual_std:.4f}
Skewness: {residual_skew:.3f}
Kurtosis: {residual_kurtosis:.3f}

NOISE COMPARISON:
Target noise std: {avg_noise:.4f}
Residual std: {residual_std:.4f}
Ratio: {ratio:.2f}

NORMALITY:
Shapiro p-value: {shapiro_p:.4f}
{"✓ Normal" if shapiro_p > 0.05 else "✗ Non-normal"}

QUALITY ASSESSMENT:
{"✓ Good fit" if 0.8 <= ratio <= 1.2 else "⚠ Check fit"}
    """
    
    ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes, 
            verticalalignment='top', fontfamily='monospace', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # Print final assessment
    print(f"\n=== FINAL MODEL ASSESSMENT ===")
    print(f"Model: {BEST_MODEL.name}")
    print(f"Individual trial success rate: {100*n_successful/len(traces_array):.1f}%")
    print(f"Residual/noise ratio: {ratio:.2f}")
    
    if 0.8 <= ratio <= 1.2:
        print("✓ EXCELLENT: Residuals match noise level well")
    elif 0.5 <= ratio <= 2.0:
        print("⚠ GOOD: Residuals reasonably match noise level")
    else:
        print("✗ POOR: Residuals don't match noise level - consider different model")
        
    if shapiro_p > 0.05:
        print("✓ EXCELLENT: Residuals are normally distributed")
    else:
        print("⚠ WARNING: Residuals are not normally distributed")

else:
    print("No successful individual fits - cannot perform residual analysis")